# 04 - Funding Pattern Clustering

Explores clusters of funding events using K-Means.

All logic lives in `src/clustering.py` so results are reproducible and testable. This notebook calls the module rather than reimplementing it; the module is the source of truth.

## 1. Load Data

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd

from src.config import CLEANED_CSV, MODELS_DIR
from src.data_loader import load_raw_csv
from src.preprocessing import preprocess
from src.clustering import (
    run_clustering,
    load_cluster_summary,
    prepare_clustering_frame,
    evaluate_k_range,
)

In [ ]:
if CLEANED_CSV.exists():
    df = pd.read_csv(CLEANED_CSV)
else:
    df = preprocess(load_raw_csv())

print(f"Dataset shape: {df.shape}")

## 2. Prepare Clustering Frame

Two deliberate choices differ from the earlier notebook-only analysis:

- Categorical inputs are **one-hot encoded**, not label encoded. Label encoding imposes an arbitrary integer ordering on cities like Bengaluru vs Mumbai, and K-Means will treat that ordering as distance.
- Funding amount is **log1p transformed**. Raw amounts have skew ~22, so unscaled dollars would collapse nearly every point onto a handful of mega-rounds, making the clusters a pure amount tiering.

In [ ]:
frame = prepare_clustering_frame(df)
print(f"Rows after dropping incomplete records: {len(frame)}")
frame.head()

## 3. Find Optimal Cluster Count

Elbow method (inertia) and silhouette score, swept over k=2..10. The silhouette maximum picks the final k.

In [ ]:
evaluation = evaluate_k_range(frame)
evaluation

## 4. Run Full Clustering

Fits the model, writes the pipeline, the JSON summary and per-event cluster assignments, and renders both report charts.

In [ ]:
result = run_clustering(df)

print(f"Optimal k: {result['optimal_k']}")
result['summary']

## 5. Cluster Profiles

In [ ]:
result['summary'][[
    'cluster', 'event_count', 'median_amount',
    'avg_year', 'top_industry', 'top_city'
]]

## 6. Load Persisted Results

Confirms the saved artifact round-trips, which is what the dashboard reads.

In [ ]:
summary = load_cluster_summary(MODELS_DIR)
print("Optimal k:", summary['optimal_k'])
print("Silhouette:", summary['silhouette'])
print("Events clustered:", summary['events_clustered'])

## 7. Interpretation

Clusters group historical funding events with similar sector, city, investment type, size and timing. They are **not** real-world startup categories and carry no forward-looking guarantee.

On the current data the model selects k=3: cluster 2 separates later, larger rounds from the smaller early-stage activity in clusters 0 and 1.